# Part 3 — Feature Engineering

## What is this notebook about?

Feature engineering is the step where we **create all the variables** that our models will use to learn. This is often the most important step in the entire project — good features make more difference than a better model.

In this notebook we will:
- Create **technical indicators**: moving averages, RSI, MACD, Bollinger Bands, volatility
- Add **macroeconomic features**: VIX, interest rates, oil, dollar, gold, Nasdaq
- Add **contextual features**: sector encoding, day of week, month
- Create our **two target variables**: next day return (regression) and direction up/down (classification)
- Apply **shift(1) on every feature** to avoid any data leakage

### The most important rule in this notebook
Every single feature must use only information available **before** the day we are trying to predict. This is why we apply `.shift(1)` everywhere — it shifts all features one day forward so the model never sees the future.


## Step 0 — Load Data & Download Additional Macro Indicators

We reload the clean dataset from notebook 01 and download 3 additional macro indicators: Gold, Nasdaq and the 2-year Treasury rate.

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import yfinance as yf
import warnings
import os

warnings.filterwarnings('ignore')
os.makedirs('data', exist_ok=True)
os.makedirs('plots', exist_ok=True)

# Load the clean dataset from notebook 01
master_df = pd.read_parquet('data/01_master_data.parquet')
macro_df  = pd.read_parquet('data/01_macro_data.parquet')

print('Dataset loaded!')
print(f'  Shape   : {master_df.shape}')
print(f'  Columns : {list(master_df.columns)}')

START_DATE = '2020-01-01'
END_DATE   = pd.Timestamp.today().strftime('%Y-%m-%d')
print(f'  Dates   : {START_DATE} to {END_DATE}')

Dataset loaded!
  Shape   : (76146, 14)
  Columns : ['date', 'ticker', 'sector', 'close', 'open', 'high', 'low', 'volume', 'return', 'VIX', 'RATE_10Y', 'SP500', 'OIL', 'DXY']
  Dates   : 2020-01-01 to 2026-03-11


We download the 3 additional macro indicators one by one to avoid timeout issues.

In [2]:
# Download Gold — safe haven asset
print('Downloading Gold...')
gold = yf.download('GC=F', start=START_DATE, end=END_DATE, auto_adjust=True, progress=False)
print(f'  Done! Shape: {gold.shape}')

  Done! Shape: (1556, 5)


In [3]:
# Download Nasdaq — tech focused index
print('Downloading Nasdaq...')
nasdaq = yf.download('^IXIC', start=START_DATE, end=END_DATE, auto_adjust=True, progress=False)
print(f'  Done! Shape: {nasdaq.shape}')

  Done! Shape: (1554, 5)


In [4]:
# Download 2-Year Treasury Rate
print('Downloading 2-Year Rate...')
rate2y = yf.download('^IRX', start=START_DATE, end=END_DATE, auto_adjust=True, progress=False)
print(f'  Done! Shape: {rate2y.shape}')

  Done! Shape: (1554, 5)


In [5]:
# Combine all new macro indicators into one DataFrame
new_macro_df = pd.DataFrame({
    'GOLD'   : gold['Close'].squeeze(),
    'NASDAQ' : nasdaq['Close'].squeeze(),
    'RATE_2Y': rate2y['Close'].squeeze()
})
new_macro_df = new_macro_df.ffill().bfill()

# Merge with existing macro data
macro_df = macro_df.join(new_macro_df, how='left').ffill().bfill()

# Compute yield curve spread: 10-year rate minus 2-year rate
# A negative spread (inverted yield curve) often signals a recession
macro_df['YIELD_SPREAD'] = macro_df['RATE_10Y'] - macro_df['RATE_2Y']

print(f'All macro indicators ready!')
print(f'  Indicators : {list(macro_df.columns)}')
print(f'  Shape      : {macro_df.shape}')
macro_df.tail(3)

All macro indicators ready!
  Indicators : ['VIX', 'RATE_10Y', 'SP500', 'OIL', 'DXY', 'GOLD', 'NASDAQ', 'RATE_2Y', 'YIELD_SPREAD']
  Shape      : (1557, 9)


,VIX,RATE_10Y,SP500,OIL,DXY,GOLD,NASDAQ,RATE_2Y,YIELD_SPREAD
Date,,,,,,,,,
2026-03-06,90.900002,98.989998,6740.020020,4.133,29.49,5146.100098,22387.679688,3.570,95.419998
2026-03-09,94.769997,99.180000,6795.990234,4.136,25.50,5091.500000,22695.949219,3.593,95.587000
2026-03-10,83.449997,98.830002,6781.479980,4.136,24.93,5229.700195,22697.099609,3.595,95.235002



## Step 1 — Technical Indicators

Now we create all the technical indicators for each stock. These are the features our models will use to detect patterns in price movements.

**Important — Anti Data Leakage Rule:**
Every indicator is computed first, then shifted by one day with `.shift(1)`. This means that on day J, the model only sees indicators computed up to day J-1. It never sees information from the day it is trying to predict.

Here are the indicators we create:
- **SMA 20 & 50**: average price over the last 20 and 50 days — identifies the trend
- **EMA 12 & 26**: same but with more weight on recent days — reacts faster to changes
- **RSI 14**: measures if the stock is overbought (>70) or oversold (<30)
- **MACD**: difference between EMA12 and EMA26 — measures momentum
- **Bollinger Bands**: upper and lower bands at 2 standard deviations — measures volatility
- **Volatility 20d**: standard deviation of returns over 20 days — measures recent risk
- **Past returns**: returns from J-1, J-5 and J-20 — gives the model a memory of recent moves

In [7]:
# This function computes all technical indicators for one stock
# We apply it to each stock separately then combine everything

def compute_technical_indicators(df):
    """
    Input: DataFrame for one stock with columns: date, close, return
    Output: same DataFrame with all technical indicators added
    """
    df = df.copy().sort_values('date')
    close = df['close']
    ret   = df['return']

    # --- Moving Averages ---
    df['SMA_20'] = close.rolling(window=20).mean()
    df['SMA_50'] = close.rolling(window=50).mean()
    df['EMA_12'] = close.ewm(span=12, adjust=False).mean()
    df['EMA_26'] = close.ewm(span=26, adjust=False).mean()

    # Price relative to moving averages (normalized)
    df['price_to_SMA20'] = close / df['SMA_20']
    df['price_to_SMA50'] = close / df['SMA_50']

    # --- RSI (14 days) ---
    delta     = close.diff()
    gain      = delta.clip(lower=0).rolling(window=14).mean()
    loss      = (-delta.clip(upper=0)).rolling(window=14).mean()
    rs        = gain / (loss + 1e-10)
    df['RSI'] = 100 - (100 / (1 + rs))

    # --- MACD ---
    df['MACD']        = df['EMA_12'] - df['EMA_26']
    df['MACD_signal'] = df['MACD'].ewm(span=9, adjust=False).mean()
    df['MACD_hist']   = df['MACD'] - df['MACD_signal']

    # --- Bollinger Bands (20 days, 2 std) ---
    sma20             = close.rolling(window=20).mean()
    std20             = close.rolling(window=20).std()
    df['BB_upper']    = sma20 + 2 * std20
    df['BB_lower']    = sma20 - 2 * std20
    df['BB_width']    = (df['BB_upper'] - df['BB_lower']) / sma20  # Normalized width
    df['BB_position'] = (close - df['BB_lower']) / (df['BB_upper'] - df['BB_lower'] + 1e-10)

    # --- Volatility ---
    df['volatility_20d'] = ret.rolling(window=20).std()

    # --- Past Returns (lags) ---
    df['return_lag1']  = ret.shift(1)
    df['return_lag5']  = ret.shift(5)
    df['return_lag20'] = ret.shift(20)

    # --- Volume features ---
    df['volume_change']   = df['volume'].pct_change()
    df['volume_MA20']     = df['volume'].rolling(window=20).mean()
    df['volume_relative'] = df['volume'] / (df['volume_MA20'] + 1e-10)

    return df

print('Function defined!')


Function defined!


In [8]:
# Apply the function to each stock separately
all_stocks_features = []

for ticker in master_df['ticker'].unique():
    stock_df = master_df[master_df['ticker'] == ticker].copy()
    stock_df = compute_technical_indicators(stock_df)
    all_stocks_features.append(stock_df)

# Combine all stocks back into one DataFrame
features_df = pd.concat(all_stocks_features, ignore_index=True)
features_df = features_df.sort_values(['ticker', 'date']).reset_index(drop=True)

print(f'Technical indicators computed for all stocks!')
print(f'  Shape   : {features_df.shape}')
print(f'  Columns : {list(features_df.columns)}')

Technical indicators computed for all stocks!
  Shape   : (76146, 35)
  Columns : ['date', 'ticker', 'sector', 'close', 'open', 'high', 'low', 'volume', 'return', 'VIX', 'RATE_10Y', 'SP500', 'OIL', 'DXY', 'SMA_20', 'SMA_50', 'EMA_12', 'EMA_26', 'price_to_SMA20', 'price_to_SMA50', 'RSI', 'MACD', 'MACD_signal', 'MACD_hist', 'BB_upper', 'BB_lower', 'BB_width', 'BB_position', 'volatility_20d', 'return_lag1', 'return_lag5', 'return_lag20', 'volume_change', 'volume_MA20', 'volume_relative']



## Etape 2 — Ajout des features macroéconomiques

Maintenant on ajoute les indicateurs macro comme features. L'idée est simple : on calcule la variation journalière de chaque indicateur (comme on l'a fait pour les rendements des actions) puis on les décale d'un jour avec `.shift(1)`.

Pourquoi utiliser la variation et pas la valeur brute ?
- La valeur brute du VIX n'est pas stationnaire (elle peut monter ou descendre sur le long terme)
- La variation journalière est stationnaire — elle fluctue autour de zéro
- C'est cohérent avec ce qu'on a fait pour les rendements des actions

On ajoute aussi le **yield spread** (taux 10 ans - taux 2 ans) qui est un indicateur économique très connu — quand il devient négatif, cela signale souvent une récession à venir.

In [9]:
# Calculer les variations journalières des indicateurs macro
macro_returns = macro_df.pct_change()
macro_returns.columns = [f'{col}_change' for col in macro_df.columns]

# Garder aussi les niveaux bruts pour certains indicateurs
# (le VIX en niveau absolu est aussi utile — un VIX à 40 est très différent d'un VIX à 15)
macro_levels = macro_df.copy()
macro_levels.columns = [f'{col}_level' for col in macro_df.columns]

# Combiner variations et niveaux
macro_all = macro_returns.join(macro_levels, how='left')
macro_all = macro_all.ffill().bfill()

# Réinitialiser l'index pour la fusion
macro_all = macro_all.reset_index().rename(columns={'Date': 'date', 'index': 'date'})
macro_all['date'] = pd.to_datetime(macro_all['date'])

# Fusionner avec notre dataset principal
features_df = features_df.merge(macro_all, on='date', how='left')
features_df[macro_all.columns[1:]] = features_df.groupby('ticker')[macro_all.columns[1:]].ffill()

print('Features macro ajoutées!')
print(f'  Shape   : {features_df.shape}')
print(f'  Nouvelles colonnes macro : {list(macro_all.columns[1:])}')

Features macro ajoutées!
  Shape   : (76146, 53)
  Nouvelles colonnes macro : ['VIX_change', 'RATE_10Y_change', 'SP500_change', 'OIL_change', 'DXY_change', 'GOLD_change', 'NASDAQ_change', 'RATE_2Y_change', 'YIELD_SPREAD_change', 'VIX_level', 'RATE_10Y_level', 'SP500_level', 'OIL_level', 'DXY_level', 'GOLD_level', 'NASDAQ_level', 'RATE_2Y_level', 'YIELD_SPREAD_level']



## Etape 3 — Features contextuelles

On ajoute maintenant des features qui donnent au modèle des informations de **contexte** sur chaque observation.

**Encodage du secteur :**
Le secteur est une variable catégorielle (texte). Les modèles ML ne savent pas travailler avec du texte, donc on le convertit en chiffres avec un **one-hot encoding** — on crée une colonne par secteur avec des 0 et des 1.

**Jour de la semaine et mois :**
En finance il existe des effets calendaires bien connus :
- L'**effet lundi** : les marchés ont tendance à baisser le lundi
- L'**effet janvier** : les marchés ont tendance à monter en janvier
- L'**effet vendredi** : les investisseurs soldent souvent leurs positions avant le weekend

Ces features permettent au modèle de tenir compte de ces patterns saisonniers.

In [10]:
# Encodage du secteur en one-hot
# Exemple: Technology → [1, 0, 0, 0, 0], Finance → [0, 1, 0, 0, 0], etc.
sector_dummies = pd.get_dummies(features_df['sector'], prefix='sector')
features_df    = pd.concat([features_df, sector_dummies], axis=1)

# Jour de la semaine (0=Lundi, 4=Vendredi)
features_df['day_of_week'] = features_df['date'].dt.dayofweek

# Mois de l'année (1=Janvier, 12=Décembre)
features_df['month'] = features_df['date'].dt.month

# Trimestre (1, 2, 3, 4)
features_df['quarter'] = features_df['date'].dt.quarter

print('Features contextuelles ajoutées!')
print(f'  Shape    : {features_df.shape}')
print(f'  Secteurs encodés : {list(sector_dummies.columns)}')
print(f'  Exemple jour de la semaine : {features_df["day_of_week"].unique()}')
print(f'  Exemple mois : {sorted(features_df["month"].unique())}')

Features contextuelles ajoutées!
  Shape    : (76146, 61)
  Secteurs encodés : ['sector_Energy', 'sector_Finance', 'sector_Healthcare', 'sector_Industrials', 'sector_Technology']
  Exemple jour de la semaine : [3 4 0 1 2]
  Exemple mois : [np.int32(1), np.int32(2), np.int32(3), np.int32(4), np.int32(5), np.int32(6), np.int32(7), np.int32(8), np.int32(9), np.int32(10), np.int32(11), np.int32(12)]



## Etape 4 — Variable cible et shift anti-leakage

C'est l'étape la plus importante du notebook. On fait deux choses ici :

**1. Créer les variables cibles :**
- `target_return` : le rendement du lendemain (J+1) — pour la régression
- `target_direction` : 1 si le prix monte demain, 0 si il baisse — pour la classification

**2. Appliquer le shift anti-leakage :**
On décale TOUTES les features d'un jour avec `.shift(1)` par stock. Cela garantit que le modèle n'utilise jamais d'information du jour qu'il essaie de prédire. C'est la règle la plus importante pour éviter le data leakage.

Sans ce shift, le modèle verrait par exemple le RSI calculé avec le prix de clôture du jour J pour prédire le rendement du jour J — ce qui est de la triche car en réalité on ne connaît pas encore ce prix au moment de prendre la décision.

In [11]:
# Liste de toutes les features qu'on va shifter
# On exclut les colonnes d'identification (date, ticker, sector) et les prix bruts
COLS_TO_SHIFT = [
    # Indicateurs techniques
    'SMA_20', 'SMA_50', 'EMA_12', 'EMA_26',
    'price_to_SMA20', 'price_to_SMA50',
    'RSI', 'MACD', 'MACD_signal', 'MACD_hist',
    'BB_upper', 'BB_lower', 'BB_width', 'BB_position',
    'volatility_20d',
    'return_lag1', 'return_lag5', 'return_lag20',
    'volume_change', 'volume_MA20', 'volume_relative',
    # Features macro
    'VIX_change', 'RATE_10Y_change', 'SP500_change',
    'OIL_change', 'DXY_change', 'GOLD_change',
    'NASDAQ_change', 'RATE_2Y_change', 'YIELD_SPREAD_change',
    'VIX_level', 'RATE_10Y_level', 'SP500_level',
    'OIL_level', 'DXY_level', 'GOLD_level',
    'NASDAQ_level', 'RATE_2Y_level', 'YIELD_SPREAD_level',
    # Features contextuelles
    'day_of_week', 'month', 'quarter'
]

print('Application du shift(1) sur toutes les features...')

# Appliquer le shift par stock (important — jamais sur tout le dataset d'un coup)
features_df[COLS_TO_SHIFT] = features_df.groupby('ticker')[COLS_TO_SHIFT].shift(1)

print('Shift appliqué!')

Application du shift(1) sur toutes les features...
Shift appliqué!


In [12]:
# Créer les variables cibles
# target_return    = rendement du LENDEMAIN (J+1)
# target_direction = 1 si hausse demain, 0 si baisse

features_df['target_return']    = features_df.groupby('ticker')['return'].shift(-1)
features_df['target_direction'] = (features_df['target_return'] > 0).astype(int)

print('Variables cibles créées!')
print(f'  target_return    : rendement J+1 (régression)')
print(f'  target_direction : direction J+1 — 1=hausse, 0=baisse (classification)')
print()
print('Distribution de target_direction:')
print(features_df['target_direction'].value_counts(normalize=True).round(3))

Variables cibles créées!
  target_return    : rendement J+1 (régression)
  target_direction : direction J+1 — 1=hausse, 0=baisse (classification)

Distribution de target_direction:
target_direction
1    0.517
0    0.483
Name: proportion, dtype: float64



## Etape 5 — Nettoyage final et sauvegarde

Maintenant qu'on a toutes nos features et nos variables cibles, on fait un dernier nettoyage avant de sauvegarder.

Les NaN restants viennent de deux sources :
- Les premières lignes de chaque stock où les fenêtres glissantes (SMA 50, volatilité 20j etc.) n'ont pas encore assez de données
- La dernière ligne de chaque stock où `target_return` est NaN car il n'y a pas de lendemain

On supprime ces lignes car elles ne peuvent pas être utilisées pour l'entraînement.

On définit aussi la liste finale des features qui sera utilisée dans tous les notebooks de modélisation.

In [13]:
# Supprimer les lignes avec des NaN dans les features ou la variable cible
rows_before = len(features_df)
features_df = features_df.dropna(subset=COLS_TO_SHIFT + ['target_return', 'target_direction'])
rows_after  = len(features_df)

print(f'Nettoyage des NaN:')
print(f'  Lignes avant : {rows_before:,}')
print(f'  Lignes après : {rows_after:,}')
print(f'  Lignes supprimées : {rows_before - rows_after:,}')
print()

# Vérification finale
print(f'NaN restants : {features_df[COLS_TO_SHIFT].isnull().sum().sum()}')
print(f'Shape finale : {features_df.shape}')

Nettoyage des NaN:
  Lignes avant : 76,146
  Lignes après : 73,647
  Lignes supprimées : 2,499

NaN restants : 0
Shape finale : (73647, 63)


In [14]:
# Définir la liste finale des features pour la modélisation
# On garde toutes les features sauf les colonnes d'identification et les prix bruts

FEATURE_COLS = COLS_TO_SHIFT + list(pd.get_dummies(features_df['sector'], prefix='sector').columns)

# Vérifier que toutes les features existent bien dans le DataFrame
FEATURE_COLS = [col for col in FEATURE_COLS if col in features_df.columns]

print(f'Features finales pour la modélisation:')
print(f'  Nombre total de features : {len(FEATURE_COLS)}')
print()
print('Liste complète:')
for i, col in enumerate(FEATURE_COLS):
    print(f'  {i+1:2d}. {col}')

Features finales pour la modélisation:
  Nombre total de features : 47

Liste complète:
   1. SMA_20
   2. SMA_50
   3. EMA_12
   4. EMA_26
   5. price_to_SMA20
   6. price_to_SMA50
   7. RSI
   8. MACD
   9. MACD_signal
  10. MACD_hist
  11. BB_upper
  12. BB_lower
  13. BB_width
  14. BB_position
  15. volatility_20d
  16. return_lag1
  17. return_lag5
  18. return_lag20
  19. volume_change
  20. volume_MA20
  21. volume_relative
  22. VIX_change
  23. RATE_10Y_change
  24. SP500_change
  25. OIL_change
  26. DXY_change
  27. GOLD_change
  28. NASDAQ_change
  29. RATE_2Y_change
  30. YIELD_SPREAD_change
  31. VIX_level
  32. RATE_10Y_level
  33. SP500_level
  34. OIL_level
  35. DXY_level
  36. GOLD_level
  37. NASDAQ_level
  38. RATE_2Y_level
  39. YIELD_SPREAD_level
  40. day_of_week
  41. month
  42. quarter
  43. sector_Energy
  44. sector_Finance
  45. sector_Healthcare
  46. sector_Industrials
  47. sector_Technology


In [15]:
# Sauvegarder le dataset final
features_df.to_parquet('data/03_features.parquet', index=False)

# Sauvegarder aussi la liste des features dans un fichier texte
with open('data/feature_cols.txt', 'w') as f:
    for col in FEATURE_COLS:
        f.write(col + '\n')

file_size = os.path.getsize('data/03_features.parquet') / 1024 / 1024

print('Fichiers sauvegardés!')
print(f'  data/03_features.parquet ({features_df.shape[0]:,} lignes x {features_df.shape[1]} colonnes — {file_size:.2f} MB)')
print(f'  data/feature_cols.txt    ({len(FEATURE_COLS)} features listées)')
print()
print('Notebook 03 terminé! Prochain notebook: 04_models_classical.ipynb')

Fichiers sauvegardés!
  data/03_features.parquet (73,647 lignes x 63 colonnes — 21.39 MB)
  data/feature_cols.txt    (47 features listées)

Notebook 03 terminé! Prochain notebook: 04_models_classical.ipynb



## Résumé — Ce qu'on a fait dans ce notebook

| Etape | Ce qu'on a fait | Pourquoi |

| Indicateurs techniques | SMA, EMA, RSI, MACD, Bollinger, Volatilité, Lags | Capturer les tendances et le momentum |
| Features macro | VIX, taux, S&P500, pétrole, dollar, or, Nasdaq, yield spread | Donner au modèle le contexte économique |
| Features contextuelles | Secteur (one-hot), jour, mois, trimestre | Capturer les effets saisonniers et sectoriels |
| Shift anti-leakage | `.shift(1)` sur toutes les features | Garantir que le modèle ne voit jamais le futur |
| Variables cibles | target_return (régression) et target_direction (classification) | Les deux choses qu'on cherche à prédire |
| Nettoyage | Suppression des NaN | Dataset propre et prêt pour la modélisation |

---
**Prochain notebook : `04_models_classical.ipynb`**
On va entraîner et comparer nos modèles classiques : Linear Regression, Random Forest et XGBoost avec walk-forward validation.